# Avance Fase 3 — Semana 2
## Núcleo algorítmico, eficiencia e implementación orientada a objetos

**Proyecto.** Factores asociados a la duración autorizada de los permisos de uso de vía
pública en San Francisco.

**Grupo 5** · Marco Álvarez Araya · María Belén Fassler Neumann · Ricardo Jaramillo Pulgar
**Curso** MCDI500 · Programación para la Ciencia de Datos · Magíster en Ciencia de Datos e
Inteligencia Artificial · Universidad Andrés Bello
**Docente** Dr. Omar Salinas Silva

---

### Qué hace esta fase

La Fase 2 dejó un *pipeline* que funciona y un conjunto procesado de 6.010 filas por 132
columnas. **Esta fase no construye un pipeline nuevo: reorganiza el mismo.** Lo que cambia es
la arquitectura del código, que pasa de celdas encadenadas a clases con estado.

El criterio para saber si la reorganización está bien hecha no es estético. Es verificable:
el pipeline de clases debe producir **exactamente** el mismo conjunto que la Fase 2. Esa
comprobación está en el apartado 8 y se ejecuta con `pd.testing.assert_frame_equal`.

### Recorrido

| § | Contenido |
|---|---|
| 1 | Contrato con las fases anteriores y carga del conjunto |
| 2 | Por qué pasar de funciones a objetos |
| 3 | La clase base `Transformador` y el contrato doble |
| 4 | Herencia: las siete clases hijas |
| 5 | Polimorfismo y la clase `Pipeline` |
| 6–7 | Etapa A y partición agrupada |
| 8 | **Verificación de equivalencia con la Fase 2** |
| 9 | La fuga de datos que la encapsulación reveló |
| 10 | La unidad de análisis: segmento o permiso |
| 11 | La transformación del objetivo, resuelta |
| 12 | Recursividad medida contra sus alternativas |
| 13 | Eficiencia: implementación propia contra scikit-learn, densa contra dispersa |
| 14 | Cohesión y acoplamiento |
| 15 | El patrón de diseño aplicado |
| 16 | Validación técnica |
| 17 | Documentación de la arquitectura y doble vía módulo/cuaderno |
| 18 | Persistencia y contrato con la Fase 4 |

### Qué se hereda de cada fase

- **De la Fase 1**: `src/utilidades.py`, `docs/diccionario_variables.csv` y
  `docs/metadatos_fase1.json`, que declaran la variable objetivo, los roles y los atributos
  prohibidos por fuga.
- **De la Fase 2**: `data/processed/permisos_procesado.csv` y `permisos_limpio.csv`, más las
  decisiones de imputación, codificación y escalamiento que aquí se encapsulan.
- **Del avance formativo**: las mediciones con `timeit`, `perf_counter` y `tracemalloc`, que
  aquí se aplican a decisiones reales del proyecto y no a ejemplos.

## Configuración

Una sola celda editable. Todo lo que depende del conjunto de datos vive aquí, de modo que
adaptar el cuaderno a otra fuente no obliga a recorrerlo entero. Los nombres replican los de
la Fase 2 para que la comparación entre ambos conjuntos sea directa.

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO — editar solo esta celda
# =====================================================================
ARCHIVO_CRUDO = "Active_Street-Use_Permits_20260909.csv"
CORTE_DECLARADO = "2026-09-09"

OBJETIVO   = "duracion_dias"        # variable a explicar
OBJETIVO_LOG = "duracion_log"       # versión transformada, decidida en §11
GRUPO_VAR  = "permit_number"        # unidad de agrupación de la partición

# Variables por rol, tal como quedaron declaradas en la Fase 2
NOMINALES  = ["tipo_permiso", "agente", "distrito", "inspector",
              "analysis_neighborhood", "permit_zipcode"]
PREFIJOS   = {"tipo_permiso": "tipo", "agente": "agente", "distrito": "distrito",
              "inspector": "insp", "analysis_neighborhood": "barrio",
              "permit_zipcode": "zip"}
NUMERICAS  = ["n_segmentos", "lag_aprob_inicio", "anio_aprobacion", "mes_inicio",
              "largo_proposito"]
BINARIAS   = ["tiene_proposito"]

CATEGORICAS_IMPUTAR = ["permit_zipcode", "Cross Street 2", "analysis_neighborhood"]
NUMERICAS_IMPUTAR   = ["lag_aprob_inicio", "anio_aprobacion"]

PROHIBIDAS = ["Status", "permit_start_date", "permit_end_date"]   # fuga del objetivo

SEMILLA = 42                        # misma que en F1 y F2
PROPORCION_PRUEBA = 0.25            # misma que en F2

PROYECTO = {
    "nombre": "Factores asociados a la duración autorizada de permisos de vía pública",
    "grupo": "Grupo 5",
    "fase": "F3",
    "integrantes": ["Marco Álvarez Araya",
                    "María Belén Fassler Neumann",
                    "Ricardo Jaramillo Pulgar"],
    "repositorio": "https://github.com/RJaramilloP/proyecto-grupo5-mcdi500",
}
# =====================================================================

print(f"Objetivo declarado : {OBJETIVO}")
print(f"Agrupación         : {GRUPO_VAR}")
print(f"Semilla            : {SEMILLA}")
print(f"Nominales          : {len(NOMINALES)}  |  numéricas: {len(NUMERICAS)}  "
      f"|  binarias: {len(BINARIAS)}")

Objetivo declarado : duracion_dias
Agrupación         : permit_number
Semilla            : 42
Nominales          : 6  |  numéricas: 5  |  binarias: 1


## Preparación del entorno

Se declaran las librerías y se registra el intérprete efectivo. La verificación del entorno
es la misma de la Fase 1: si el cuaderno corre fuera del entorno virtual, lo dice en la
salida en lugar de fallar más adelante por una versión distinta.

`capturar()` reaparece aquí desde la Fase 2. Ejecuta una función, muestra su salida y además
la devuelve como texto, de modo que los anexos del informe se generan desde la propia
ejecución y no se transcriben a mano.

In [2]:
import contextlib
import io
import json
import math
import pickle
import platform
import sys
import time
import timeit
import tracemalloc
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

np.random.seed(SEMILLA)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)


def capturar(funcion, *args, **kwargs):
    """Ejecuta una función, muestra su salida y además la devuelve como texto."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        resultado = funcion(*args, **kwargs)
    texto = buffer.getvalue()
    print(texto, end="")
    return resultado, texto


def verificar_entorno():
    """Registra intérprete, plataforma y versiones instaladas."""
    en_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
    print(f"Intérprete      : {sys.executable}")
    print(f"Entorno virtual : {'[OK] activo' if en_venv else '[AVISO] parece el Python del sistema'}")
    print(f"Carpeta         : {Path.cwd()}")
    print(f"Sistema         : {platform.system()} {platform.release()}")
    print(f"Python          : {platform.python_version()}")
    print("\nLibrerías declaradas")
    import sklearn
    import scipy
    for mod in (np, pd, matplotlib, sklearn, scipy):
        print(f"  [OK]  {mod.__name__:<12} {mod.__version__}")


_, ANEXO_A3 = capturar(verificar_entorno)

Intérprete      : /usr/bin/python3
Entorno virtual : [AVISO] parece el Python del sistema
Carpeta         : /home/claude/limpio/F3
Sistema         : Linux 6.18.44-fc-v50
Python          : 3.11.15

Librerías declaradas
  [OK]  numpy        2.0.2
  [OK]  pandas       2.3.3
  [OK]  matplotlib   3.9.4
  [OK]  sklearn      1.6.1
  [OK]  scipy        1.13.1


## 1. Contrato con las fases anteriores

La Fase 2 estableció un contrato formal: importaba de la Fase 1 el diccionario y los
metadatos, y se detenía con una excepción si faltaba cualquiera de los dos. La coherencia
entre fases dejaba de ser una afirmación del informe y pasaba a ser una condición que el
código verifica.

**Esta fase extiende ese contrato a tres fuentes**, porque ahora hay más que heredar: el
diccionario y los metadatos de la Fase 1, los metadatos y el conjunto procesado de la Fase 2,
y el archivo original intacto. Si falta cualquiera, el proceso se detiene aquí y no produce
resultados parciales que después habría que rastrear.

In [3]:
MARCADORES = (".git", "requirements.txt", ".gitignore")


def localizar_raiz(inicio=None, marcadores=MARCADORES, niveles=5, respaldo=True):
    """Resuelve la raíz del proyecto subiendo hasta encontrar un marcador de repositorio.

    Evita rutas absolutas y `os.chdir()`: el cuaderno produce el mismo árbol de salidas
    sin importar desde qué carpeta se lance.
    """
    actual = Path(inicio or Path.cwd()).resolve()
    for candidata in [actual, *actual.parents][:niveles + 1]:
        if any((candidata / m).exists() for m in marcadores):
            return candidata, "estructurado"
    if respaldo:
        return actual, "plano"
    raise FileNotFoundError(f"No se encontró marcador de repositorio desde {actual}.")


RAIZ, MODO_RUTAS = localizar_raiz()
DIR_DATOS      = RAIZ / "data"
DIR_CRUDO      = DIR_DATOS / "raw"
DIR_PROCESADO  = DIR_DATOS / "processed"
DIR_DOCS       = RAIZ / "docs"
DIR_ANEXOS     = DIR_DOCS / "anexos"
DIR_SRC        = RAIZ / "src"
DIR_F3         = RAIZ / "F3"
DIR_ARTEFACTOS = RAIZ / "artefactos"

for carpeta in (DIR_PROCESADO, DIR_DOCS, DIR_ANEXOS, DIR_SRC, DIR_F3, DIR_ARTEFACTOS):
    carpeta.mkdir(parents=True, exist_ok=True)

if str(DIR_SRC) not in sys.path:
    sys.path.insert(0, str(DIR_SRC))
import utilidades

print(f"Raíz del proyecto : {RAIZ}")
print(f"Modo de rutas     : {MODO_RUTAS}")
print(f"Módulo compartido : utilidades desde {Path(utilidades.__file__).relative_to(RAIZ).as_posix()}")

Raíz del proyecto : /home/claude/limpio
Modo de rutas     : estructurado
Módulo compartido : utilidades desde src/utilidades.py
